# 04. EuroCrops collection

Use the local 2021 parcel declarations to collect annual Sentinel-2 imagery for development. CropHarvest remains the reserved test source. Run the relevant source top-ups, then run collection 06 and preprocessing 05–07.

This notebook keeps the existing reference and imagery paths, campaign IDs, top-up settings, and progress logs. The crop-specific setup applies to new requests; earlier batches retain their saved plans. Organized outputs are linked under `pipeline_outputs/collection/eurocrops/`.


## 1. Setup and configuration

Review the local parcel path, batch size, and collection mode before running.

In [ ]:
# Review these settings before Run All.
import os
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
from crop_pipeline import notebook_setup

run_satellite_collection = True
submit_only = False  # Queue exports. Set False to collect finished results.
extra_batches = 21  # Total extra batch slots for this round.
parcels_per_batch = 100
parcel_path = root / "data/external/eurocrops_v2/NL_2021.parquet"
notebook_setup.prepare_environment(root, install_missing=run_satellite_collection)
os.chdir(root)
settings_path = root / "settings.local.json"
settings = notebook_setup.ensure_local_settings(settings_path)
report_root = (root / Path(settings.get("report_root", "sen4agrinet_work/inventory")).expanduser()).resolve()


Notebook environment ready: /Users/josedavid/Documents/programming/external-crop-data-classification/.venv/bin/python


## 2. Obtain references

Use the local Dutch EuroCrops parcel declarations for 2021.

In [ ]:
from crop_pipeline.sources import adaptive, annual, eurocrops
from crop_pipeline.windows.seasons import file_sha256

print("Local EuroCrops parcel file:", parcel_path)


Local EuroCrops parcel file: /Users/josedavid/Documents/programming/external-crop-data-classification/data/external/eurocrops_v2/NL_2021.parquet


## 3. Validate references

Select declared carrot production parcels with valid footprints.

In [ ]:
features = eurocrops.load_carrots(parcel_path)
print("Eligible declared carrot references:", len(features))


Eligible declared carrot references: 2194


## 4. Prepare the collection round

Resume saved plans and select unused carrot parcels for new batches.

In [ ]:
batches = adaptive.prepare_round_batches(
    eurocrops.SOURCE, features, annual.archive_root(settings, report_root), report_root,
    provenance={"dataset": "EuroCrops", "country": "NL", "year": eurocrops.YEAR,
                "parcel_file_sha256": file_sha256(parcel_path),
                "carrot_production_codes": sorted(eurocrops.CARROT_PRODUCTION_CODES)},
    parcels_per_crop=parcels_per_batch, candidate_crops=("carrot",), extra_batches=extra_batches,
)


Round 2: resuming saved eurocrops selections and original collection plans.
Round 2: up to 100 candidates per crop in each new batch. Resume/submit 6 batches.


## 5. Collect annual imagery

Use `submit_only = True` to queue exports. Use `False` to download finished results.

In [ ]:
from crop_pipeline import source_workflow

project = notebook_setup.project_for_collection(settings, settings_path, required=run_satellite_collection)
result = adaptive.collect_batches(
    batches, project, enabled=run_satellite_collection, authenticate=run_satellite_collection,
    submit_only=submit_only, download_workers=2, intervals_per_request=6,
)


eurocrops 2021: selecting tiles 1–139 of 139
Scene inventories: 25/139 (0.0s)
Scene inventories: 50/139 (0.0s)
Scene inventories: 75/139 (0.1s)
Scene inventories: 100/139 (0.1s)
Scene inventories: 125/139 (0.1s)
Scene inventories: 139/139 (0.1s)
  Resuming with 18,000 rows from 18 verified cached pages.
  Fetching page 19 (up to 5,000 rows; 18,000 saved so far)…
  Page 19 verified: 1,000 rows; 19,000 total (0.6s).
  Fetching page 20 (up to 5,000 rows; 19,000 saved so far)…
  Page 20 verified: 1,000 rows; 20,000 total (0.7s).
  Fetching page 21 (up to 5,000 rows; 20,000 saved so far)…
  Page 21 verified: 1,000 rows; 21,000 total (0.8s).
  Fetching page 22 (up to 5,000 rows; 21,000 saved so far)…
  Page 22 verified: 1,000 rows; 22,000 total (0.6s).
  Fetching page 23 (up to 5,000 rows; 22,000 saved so far)…
  Page 23 verified: 1,000 rows; 23,000 total (0.8s).
  Fetching page 24 (up to 5,000 rows; 23,000 saved so far)…
  Page 24 verified: 1,000 rows; 24,000 total (1.6s).
  Fetching page 2

## 6. Summary and next action

Check the batch status below. Once collection is complete, run collection 06, then preprocessing 05–07 to include EuroCrops in the development features.

In [ ]:
source_workflow.print_summary(eurocrops.SOURCE, batches, result, "reuse", 10)


eurocrops — completed
References: reuse | Inward buffer: 10 m
Selected parcels: carrot 600
Batches: eurocrops_extra_0015: completed; eurocrops_extra_0016: completed; eurocrops_extra_0017: completed; eurocrops_extra_0018: completed; eurocrops_extra_0019: completed; eurocrops_overnight_100: completed
Verified local tiles: 808/808 (reported batch ranges)
Cloud jobs: 0 queued, 0 running, 0 ready to download, 0 failed, 0 cancelled, 0 unknown
Campaign: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/adaptive_training/829e0cec79b8
Annual TIFF root: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_tifs
Saved registries: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/adaptive_training/829e0cec79b8/rounds/0002/eurocrops_extra_0015/source_handoffs/full_year_all_parcels/eurocrops.json, /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrin

## Organized outputs

Refresh the linked output folders without moving downloads or changing saved jobs. Run **06_dataset_summary.ipynb** to record all saved batches and selections. Historical outputs above were retained from the original notebook; rerun deliberately to update them.


In [ ]:
from crop_pipeline.output_layout import refresh

output_layout = refresh(root)
print("Organized collection outputs:", root / "pipeline_outputs/collection")
print("Next: data_collection/06_dataset_summary.ipynb, then preprocessing/01_imagery_quality.ipynb.")


Organized collection outputs: /Users/josedavid/Documents/programming/external-crop-data-classification/pipeline_outputs/collection
Next: data_collection/06_dataset_summary.ipynb, then preprocessing/01_imagery_quality.ipynb.
